# Parameter tuning with GridSearchCV to see if we can easily improve results

While doing research, we learned that we could automatically tune model parameters using GridSearchCV from sklearn. This allows us to specify a grid of parameters to search over, and the model will be trained and evaluated for each combination of parameters in the grid. This can help us find the best set of parameters for our models and potentially improve their performance, without us having to do anything.

So we will see if this can make any improvements before we continue.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import GridSearchCV

In [2]:
df = pd.read_csv("data/extra_and_merged_data.csv", index_col="date", parse_dates=True)
df.sort_index(inplace=True)

In [3]:
# Defining features and target variable
X = df[["summary", "pct_change", "volume"]]
y = df["target"]

In [4]:
#creating training and testing sets without shuffling to maintain temporal order
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [5]:
print(f"Training Range: {X_train.index.min()} to {X_train.index.max()}")
print(f"Testing Range:  {X_test.index.min()} to {X_test.index.max()}")

Training Range: 2003-02-20 00:00:00 to 2018-03-23 00:00:00
Testing Range:  2018-03-26 00:00:00 to 2021-12-30 00:00:00


In [6]:
# Defining features
text_features = "summary"
numerical_features = ["pct_change", "volume"]

In [7]:

# creating preprocessors for each model
preprocessor_sgd = ColumnTransformer(
    transformers=[
        (
            "text",
            TfidfVectorizer(max_features=5000, ngram_range=(1, 2)),
            text_features,
        ),
        ("num", StandardScaler(), numerical_features),
    ]
)

preprocessor_lda = ColumnTransformer(
    transformers=[
        # Reduce text to 50 components using SVD (PCA for text)
        (
            "text",
            Pipeline(
                [
                    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
                    ("svd", TruncatedSVD(n_components=50)),
                ]
            ),
            text_features,
        ),
        ("num", StandardScaler(), numerical_features),
    ]
)

preprocessor_mnb = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(max_features=5000, ngram_range=(1, 2)),
            text_features,
        ),
        ("scaler", MinMaxScaler(clip=True), numerical_features),
    ]
)

In [8]:
# Creating pipelines for each model
pipeline_sgd = Pipeline([
    ('prep', preprocessor_sgd),
    ('clf', SGDClassifier(loss='log_loss', penalty='l2', max_iter=1000, random_state=42))
])

pipeline_lda = Pipeline([
    ('prep', preprocessor_lda),
    ('clf', LinearDiscriminantAnalysis())
])

pipeline_mnb = Pipeline([
    ('prep', preprocessor_mnb),
    ('clf', MultinomialNB(alpha=0.1, fit_prior=False))
])


I am using gridsearch to find the best parameters for each model.

In [9]:
# Setting up GridSearchCV for each model to tune hyperparameters
grid_sgd = GridSearchCV(
    estimator=pipeline_sgd,
    param_grid={
        "clf__alpha": [1e-4, 1e-3, 1e-2],
        "clf__loss": ["hinge", "log_loss"]
    },
    cv=3,
    n_jobs=-1,
    scoring="accuracy"
)

grid_sgd.fit(X_train, y_train)


grid_lda = GridSearchCV(
    estimator=pipeline_lda,
    param_grid={
        "clf__solver": ["svd", "lsqr"]
    },
    cv=3,
    n_jobs=-1,
    scoring="accuracy"
)

grid_lda.fit(X_train, y_train)

grid_mnb = GridSearchCV(
    estimator=pipeline_mnb,
    param_grid={
        "clf__alpha": [0.1, 0.5, 1.0]
    },
    cv=3,
    n_jobs=-1,
    scoring="accuracy"
)

grid_mnb.fit(X_train, y_train)


,estimator,Pipeline(step...rior=False))])
,param_grid,"{'clf__alpha': [0.1, 0.5, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('tfidf', ...), ('scaler', ...)]"


Printing the results

In [10]:
print("SGD Best Params:", grid_sgd.best_params_)
print("SGD Test Acc:", grid_sgd.score(X_test, y_test))

y_pred_sgd = grid_sgd.predict(X_test)

print("SGD Classification Report:")
print(confusion_matrix(y_test, y_pred_sgd))
print(classification_report(y_test, y_pred_sgd))

SGD Best Params: {'clf__alpha': 0.01, 'clf__loss': 'hinge'}
SGD Test Acc: 0.5648050579557429
SGD Classification Report:
[[  1 412]
 [  1 535]]
              precision    recall  f1-score   support

         0.0       0.50      0.00      0.00       413
         1.0       0.56      1.00      0.72       536

    accuracy                           0.56       949
   macro avg       0.53      0.50      0.36       949
weighted avg       0.54      0.56      0.41       949



The tuned SGD model reached 56% accuracy, but it almost always predicts class 1 (99% recall) and barely recognizes class 0, showing extreme imbalance in its predictions.

In [11]:
print("LDA Best Params:", grid_lda.best_params_)
print("LDA Test Acc:", grid_lda.score(X_test, y_test))

y_pred_lda = grid_lda.predict(X_test)

print("LDA Classification Report:")
print(confusion_matrix(y_test, y_pred_lda))
print(classification_report(y_test, y_pred_lda))

LDA Best Params: {'clf__solver': 'svd'}
LDA Test Acc: 0.5500526870389885
LDA Classification Report:
[[ 81 332]
 [ 95 441]]
              precision    recall  f1-score   support

         0.0       0.46      0.20      0.28       413
         1.0       0.57      0.82      0.67       536

    accuracy                           0.55       949
   macro avg       0.52      0.51      0.47       949
weighted avg       0.52      0.55      0.50       949



The tuned LDA model achieved 56% accuracy, performing very well on class 1 (86% recall) but poorly on class 0, indicating a strong bias toward predicting class 1.

In [12]:

print("MNB Best Params:", grid_mnb.best_params_)
print("MNB Test Acc:", grid_mnb.score(X_test, y_test))

y_pred_mnb = grid_mnb.predict(X_test)

print("MNB Classification Report:")
print(confusion_matrix(y_test, y_pred_mnb))
print(classification_report(y_test, y_pred_mnb))

MNB Best Params: {'clf__alpha': 1.0}
MNB Test Acc: 0.46680716543730244
MNB Classification Report:
[[236 177]
 [329 207]]
              precision    recall  f1-score   support

         0.0       0.42      0.57      0.48       413
         1.0       0.54      0.39      0.45       536

    accuracy                           0.47       949
   macro avg       0.48      0.48      0.47       949
weighted avg       0.49      0.47      0.46       949



The tuned Multinomial Naive Bayes model achieved 47% accuracy, struggling to distinguish classes it predicts class 0 correctly 58% of the time but performs poorly on class 1 (40% recall), showing overall weak classification.

In [13]:
%%capture
%pip install joblib

In [14]:
# exporting the baseline model to compare after we tune it
import joblib

joblib.dump(pipeline_sgd, "models/sgd_baseline_more_data.pkl")
joblib.dump(pipeline_lda, "models/lda_baseline_more_data.pkl")
joblib.dump(pipeline_mnb, "models/mnb_baseline_more_data.pkl")

['models/mnb_baseline_more_data.pkl']

the models LDA have a pitfall which is causing some underlying issues. The model pretty much always guess up as they have noticed the trend of it rising. It has learned that guessing up is safer. the Naives Bayes came out to worse than random guessing. SGD has same issue of LDA but slightly more balance on its up vs down guesses but it is still heavily guessing up.

Our low F1-score is proving that the model is not actually learning market mechanics it is just guessing based of the trend. We are also drowning out our financial data with our news being a majority. So to combat that we are going to use the news to create a sentiment value and then pass that and the finical data to a model. 

## Testing for overfitting within our models

In [15]:
from sklearn.model_selection import cross_val_score

def evaluate_overfitting(model, X_train, y_train, X_test, y_test, name="Model"):
    # Training accuracy
    train_acc = model.score(X_train, y_train)

    # Cross-validation accuracy (3-fold)
    cv_scores = cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy")

    # Test accuracy
    test_acc = model.score(X_test, y_test)

    print(f"\n=== {name} Overfitting Analysis ===")
    print(f"Training Accuracy:      {train_acc:.4f}")
    print(f"CV Mean Accuracy:       {cv_scores.mean():.4f}  (± {cv_scores.std():.4f})")
    print(f"Test Accuracy:          {test_acc:.4f}")

    if train_acc - test_acc > 0.10:
        print("Potential OVERFITTING detected (training >> test).")
    elif test_acc - train_acc > 0.10:
        print("Potential UNDERFITTING (model too weak).")
    else:
        print("Model generalizes well.")


In [16]:
evaluate_overfitting(grid_sgd, X_train, y_train, X_test, y_test, name="SGD")
evaluate_overfitting(grid_lda, X_train, y_train, X_test, y_test, name="LDA")
evaluate_overfitting(grid_mnb, X_train, y_train, X_test, y_test, name="MNB")



=== SGD Overfitting Analysis ===
Training Accuracy:      0.5476
CV Mean Accuracy:       0.5481  (± 0.0010)
Test Accuracy:          0.5648
Model generalizes well.

=== LDA Overfitting Analysis ===
Training Accuracy:      0.5565
CV Mean Accuracy:       0.5328  (± 0.0087)
Test Accuracy:          0.5501
Model generalizes well.

=== MNB Overfitting Analysis ===
Training Accuracy:      0.7436
CV Mean Accuracy:       0.5188  (± 0.0254)
Test Accuracy:          0.4668
Potential OVERFITTING detected (training >> test).


SGD training and testing are basically identical so there is no overfitting. This is good.

LDA is the same so there is no overfitting. Also good.

MNB the training accuracy is alot higher than the testing accuracy leading to overfitting. This is bad.